# 실습 5 · Self Distillation

경량 ResNet-18 **하나**의 `layer2`에 Intermediate Classifier를 연결합니다. 같은 모델의 Final Classifier가 만든 확률을 Intermediate Classifier의 추가 학습 신호로 사용합니다. [Be Your Own Teacher](https://arxiv.org/abs/1905.08094)의 아이디어를 단순화한 실습입니다.

## 1. 실행 환경과 CIFAR-10

처음부터 순서대로 실행합니다. CUDA가 있으면 GPU를 사용하며, CIFAR-10은 캐시가 없는 런타임에서 내려받습니다.

기본 실습은 **학습 64배치, 평가 512장**입니다. 난수 시드를 고정해 테스트 샘플을 선택하고, 학습 배치와 평가 데이터를 분리합니다.

**코드 셀 1 · 라이브러리와 실습 설정**

라이브러리를 불러오고 실행 장치, 데이터·가중치 경로, 무작위 시드, 학습 64배치와 테스트 512장 설정을 정합니다.

In [ ]:
# 실습 설정을 한곳에서 관리한다. 데이터·가중치 캐시를 재사용하고 CUDA가 있으면 GPU로 실행한다.
import copy
import os
import time
from itertools import islice
from pathlib import Path
from statistics import mean

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# 동일한 시드와 배치 수를 사용해 반복 실행 조건을 맞춘다.
SEED = 50
BATCH_SIZE = 64
TRAIN_STEPS = 64
TEST_SAMPLES = 512
DATA_ROOT = Path(os.environ.get("AX_IDEC_DATA_ROOT", "./data"))
ASSET_ROOT = Path(os.environ.get("AX_IDEC_CACHE", "./idec_cache"))
ASSET_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("실행 장치:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU")

**코드 셀 2 · CIFAR-10 데이터 준비**

CIFAR-10의 학습·테스트 데이터를 내려받아 RGB 채널을 정규화하고, 고정된 샘플을 64장씩 전달하는 Loader를 만듭니다.

이미 내려받은 CIFAR-10은 런타임의 캐시에서 읽습니다. 첫 실행에서는 공유 Google Drive에서 받고, 실패하면 CIFAR-10의 공식 주소를 사용합니다.

In [ ]:
# CIFAR-10 학습·테스트 분할에 같은 RGB 정규화를 적용한다. 테스트는 학습에 쓰이지 않는다.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
from http.client import HTTPException
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from urllib.request import urlopen

from torchvision.datasets.utils import check_integrity


class DriveCIFAR10(datasets.CIFAR10):
    # 압축 해제된 데이터나 검증된 압축 파일을 먼저 재사용한다.
    def download(self):
        if self._check_integrity():
            return

        root = Path(self.root)
        archive = root / self.filename
        if not check_integrity(archive, self.tgz_md5):
            drive_url = (
                "https://drive.usercontent.google.com/download"
                "?id=1Sx382Bt_XcAUj9_igoCgWoNQ6c5ooSa6&export=download&confirm=t"
            )
            try:
                root.mkdir(parents=True, exist_ok=True)
                with TemporaryDirectory(dir=root) as temp_dir:
                    staged = Path(temp_dir) / self.filename
                    print("Google Drive에서 CIFAR-10을 내려받습니다.")
                    with urlopen(drive_url, timeout=30) as response:
                        if response.headers.get_content_type() == "text/html":
                            raise ValueError("Drive가 파일 대신 안내 페이지를 반환했습니다.")
                        with staged.open("wb") as output:
                            copyfileobj(response, output, length=1024 * 1024)
                    if not check_integrity(staged, self.tgz_md5):
                        raise ValueError("CIFAR-10 원본 체크섬과 일치하지 않습니다.")
                    staged.replace(archive)
            except (OSError, HTTPException, ValueError):
                print("Drive에서 CIFAR-10을 받지 못해 공식 다운로드 주소로 다시 시도합니다.")

        # torchvision이 원본 체크섬을 확인하고 필요한 경우 압축을 해제한다.
        super().download()


train_data = DriveCIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_data = DriveCIFAR10(DATA_ROOT, train=False, download=True, transform=transform)

# 시드를 고정해 일부 샘플만 선택하고, 같은 데이터에서 반복 비교할 Loader를 만든다.
def make_loader(dataset, samples, seed):
    indices = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(seed))
    subset = Subset(dataset, indices[:samples].tolist())
    return DataLoader(subset, batch_size=BATCH_SIZE, num_workers=0,
                      pin_memory=(DEVICE.type == "cuda"))

train_loader = make_loader(train_data, (TRAIN_STEPS + 1) * BATCH_SIZE, SEED)
test_loader = make_loader(test_data, TEST_SAMPLES, SEED + 1)
print(f"학습 {TRAIN_STEPS} batches / 평가 {len(test_loader.dataset)} images")

## 2. 공통 Student 모델

경량 ResNet-18은 BasicBlock의 공간 연산을 Depthwise 3×3과 Pointwise 1×1로 나눕니다. 입력은 32×32 RGB 영상이고, 분류기는 CIFAR-10 클래스별 Logit 10개를 출력합니다.

학습 방법에 따라 달라지는 것은 **Student가 정답 외에 함께 받는 학습 신호**입니다.

**코드 셀 3 · 경량 ResNet-18 정의**

BasicBlock의 첫 3×3 연산을 Depthwise와 Pointwise로 나눈 경량 ResNet-18 모델의 생성 함수를 정의합니다.

In [ ]:
# 기존 BasicBlock의 3×3 연산을 Depthwise와 Pointwise로 나누어 경량 모델을 만든다.
def build_light_resnet18():
    __all__ = ['ResNet', 'resnet18']

    def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
        """3x3 convolution with padding"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                         padding=dilation, groups=in_planes, bias=False, dilation=dilation)

    def conv1x1(in_planes, out_planes, stride=1):
        """1x1 convolution"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

    # 주 경로에서 특징을 계산하고 Residual 입력을 더한다. 크기가 바뀌는 블록에서는 Residual도 변환한다.
    class BasicBlock(nn.Module):
        expansion = 1

        def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                     base_width=64, dilation=1, norm_layer=None):
            super(BasicBlock, self).__init__()

            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            if groups != 1 or base_width != 64:
                raise ValueError(
                    'BasicBlock only supports groups=1 and base_width=64')
            if dilation > 1:
                raise NotImplementedError(
                    "Dilation > 1 not supported in BasicBlock")
            # Both self.conv1 and self.downsample layers downsample the input when stride != 1
            # 3×3 Depthwise에서 채널별 공간 특징을 계산하고, 1×1 Pointwise에서 채널을 조합한다.
            self.dwconv33_1 = conv3x3(inplanes, planes, stride)
            self.conv11_1 = conv1x1(planes, planes)
            self.bn1 = norm_layer(planes)
            self.relu = nn.ReLU(inplace=True)

            self.dwconv33_2 = conv3x3(planes, planes)
            self.conv11_2 = conv1x1(planes, planes)
            self.bn2 = norm_layer(planes)
            self.downsample = downsample
            self.stride = stride

        def forward(self, x):
            identity = x

            out = self.dwconv33_1(x)
            out = self.conv11_1(out)
            out = self.bn1(out)
            out = self.relu(out)

            out = self.dwconv33_2(out)
            out = self.conv11_2(out)
            out = self.bn2(out)

            if self.downsample is not None:
                identity = self.downsample(x)

            out += identity
            out = self.relu(out)

            return out

    # 입력 stem과 Residual stage를 연결하고, Global Average Pooling 뒤에서 클래스별
    # Logit을 계산한다.
    class ResNet(nn.Module):

        def __init__(self, block, layers, num_classes=1000, zero_init_residual=False,
                     groups=1, width_per_group=64, replace_stride_with_dilation=None,
                     norm_layer=None):
            super(ResNet, self).__init__()
            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            self._norm_layer = norm_layer

            self.inplanes = 64
            self.dilation = 1
            if replace_stride_with_dilation is None:
                # each element in the tuple indicates if we should replace
                # the 2x2 stride with a dilated convolution instead
                replace_stride_with_dilation = [False, False, False]
            if len(replace_stride_with_dilation) != 3:
                raise ValueError("replace_stride_with_dilation should be None "
                                 "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
            self.groups = groups
            self.base_width = width_per_group
            self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3,
                                   bias=False)

            self.bn1 = norm_layer(self.inplanes)
            self.relu = nn.ReLU(inplace=True)
            self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            # 네 stage는 64·128·256·512채널을 사용한다. 첫 블록의 stride로 해상도를 조정한다.
            self.layer1 = self._make_layer(block, 64, layers[0])
            self.layer2 = self._make_layer(block, 128, layers[1], stride=2,
                                           dilate=replace_stride_with_dilation[0])
            self.layer3 = self._make_layer(block, 256, layers[2], stride=2,
                                           dilate=replace_stride_with_dilation[1])
            self.layer4 = self._make_layer(block, 512, layers[3], stride=2,
                                           dilate=replace_stride_with_dilation[2])
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
            self.fc = nn.Linear(512 * block.expansion, num_classes)

            # Convolution과 정규화 계층의 학습 초기값을 설정한다.
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        m.weight, mode='fan_out', nonlinearity='relu')
                elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                    nn.init.constant_(m.weight, 1)
                    nn.init.constant_(m.bias, 0)

            # Zero-initialize the last BN in each residual branch,
            # so that the residual branch starts with zeros, and each residual block behaves like an identity.
            # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
            if zero_init_residual:
                for m in self.modules():
                    if isinstance(m, Bottleneck):
                        nn.init.constant_(m.bn3.weight, 0)
                    elif isinstance(m, BasicBlock):
                        nn.init.constant_(m.bn2.weight, 0)

        # 첫 블록에서 필요한 Residual 변환을 적용하고, 뒤의 블록은 같은 출력 크기로 이어 붙인다.
        def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
            norm_layer = self._norm_layer
            downsample = None
            previous_dilation = self.dilation
            if dilate:
                self.dilation *= stride
                stride = 1
            if stride != 1 or self.inplanes != planes * block.expansion:
                downsample = nn.Sequential(
                    conv1x1(self.inplanes, planes * block.expansion, stride),
                    norm_layer(planes * block.expansion),
                )

            layers = []
            layers.append(block(self.inplanes, planes, stride, downsample, self.groups,
                                self.base_width, previous_dilation, norm_layer))
            self.inplanes = planes * block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, planes, groups=self.groups,
                                    base_width=self.base_width, dilation=self.dilation,
                                    norm_layer=norm_layer))

            return nn.Sequential(*layers)

        # 입력에서 stem, 네 stage, Pooling, 최종 Linear Layer로 이어지는 순전파다.
        def _forward_impl(self, x):
            # See note [TorchScript super()]
            x = self.conv1(x)
            x = self.bn1(x)
            x = self.relu(x)
            x = self.maxpool(x)

            x = self.layer1(x)
            x = self.layer2(x)
            x = self.layer3(x)
            x = self.layer4(x)

            x = self.avgpool(x)
            x = torch.flatten(x, 1)
            x = self.fc(x)

            return x

        def forward(self, x):
            return self._forward_impl(x)

    # CIFAR-10에 맞게 첫 Conv2d를 3×3, stride 1로 설정하고 출력 클래스 수를 10으로 둔다.
    def resnet_18_cifar_lightweight():
        r"""
            Modify the resnet 18 network in order to run on cifar-10 dataset

            To enhance the accuracy, the (kernel_size, stride, padding) of conv1 is modified to (3, 1, 1)
            referenced by <https://github.com/akamaster/pytorch_resnet_cifar10>
        """
        model = ResNet(block=BasicBlock, layers=[2, 2, 2, 2], num_classes=10)

        model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1,
                                bias=False)

        return model

    return resnet_18_cifar_lightweight()

## 3. Intermediate Classifier 추가

한 영상에서 나온 `layer2` Feature Map을 두 경로가 함께 사용합니다.

- **Intermediate Classifier** · 128채널 중간 특징을 Global Average Pooling으로 줄이고 Linear Layer에서 10개 Logit 계산
- **Final Classifier** · 같은 중간 특징을 `layer3`, `layer4`까지 보내고 기존 최종 분류기에서 10개 Logit 계산

두 출력은 같은 모델에서 나오지만 **서로 다른 분류기**가 계산합니다.

**코드 셀 4 · 중간 Feature Map 추출과 분류**

입력에서 layer2의 Feature Map을 꺼내고 이 특징을 나머지 층과 최종 분류기에 전달하는 함수를 정의합니다.

In [ ]:
# 입력에서 layer2까지의 중간 특징을 꺼낸다. 같은 Feature Map을 분류 계산에도 재사용할 수 있다.
def features_at_layer2(model, images):
    x = model.conv1(images)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)
    x = model.layer1(x)
    return model.layer2(x)

# 중간 Feature Map을 후속 stage와 최종 분류기에 전달한다.
def predict_from_layer2(model, feature):
    x = model.layer3(feature)
    x = model.layer4(x)
    x = torch.flatten(model.avgpool(x), 1)
    return model.fc(x)

**코드 셀 5 · Response KD Loss 정의**

Temperature를 적용한 Student와 목표 모델의 확률 분포 사이 KL divergence에 Temperature의 제곱을 곱해 손실을 계산합니다.

In [ ]:
# Student 로그 확률과 고정된 목표 확률의 KL divergence에 T²를 곱해 KD Loss를 계산한다.
def response_kd(student_logits, teacher_logits, temperature):
    log_student = F.log_softmax(student_logits / temperature, dim=1)
    soft_target = F.softmax(teacher_logits.detach() / temperature, dim=1)
    return F.kl_div(log_student, soft_target, reduction="batchmean") * temperature**2

**코드 셀 6 · 두 분류기를 갖는 ResNet과 비교군 준비**

layer2 뒤의 Intermediate 분류기와 원래 Final 분류기를 갖는 ResNet을 정의하고 정답 학습용과 Self KD용 모델을 같은 가중치로 초기화합니다.

In [ ]:
# 모든 Logit을 같은 양수 T로 나눈 뒤 Softmax를 적용한다. 함께 둔 계수는 KD 손실의 비중을 정한다.
TEMPERATURE = 2.0
SELF_WEIGHT = 0.5

class TwoClassifierResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = build_light_resnet18()
        self.intermediate = nn.Linear(128, 10)

    # 중간 Feature Map을 Global Average Pooling으로 줄여 중간 분류기에 전달한다.
    def classify_intermediate(self, feature):
        return self.intermediate(feature.mean(dim=(2, 3)))

    def forward(self, images):
        feature = features_at_layer2(self.backbone, images)
        middle = self.classify_intermediate(feature)
        final = predict_from_layer2(self.backbone, feature)
        return middle, final

# 두 분류기 구조를 가진 기준 모델을 만들고 같은 초기 가중치로 Self KD 모델을 준비한다.
label_only = TwoClassifierResNet().to(DEVICE)
self_kd = copy.deepcopy(label_only)
optim_label = torch.optim.SGD(label_only.parameters(), lr=0.03, momentum=0.9)
optim_self = torch.optim.SGD(self_kd.parameters(), lr=0.03, momentum=0.9)

## 4. 정답과 Final Classifier 출력 사용

비교하는 두 모델 모두 Intermediate와 Final Classifier를 정답으로 학습합니다. Self KD 모델에서는 Intermediate에 **Final Classifier의 예측 확률을 맞추는 손실**을 추가합니다. 이 손실에서 Final의 출력은 고정된 Soft Target으로 사용합니다.

`response_kd(middle, final, ...)`에서 앞쪽 인수는 학습할 출력이고 뒤쪽 인수는 목표 출력입니다.

**코드 셀 7 · 정답 학습과 Self KD 학습**

같은 배치에서 두 모델을 학습하되 Self KD 모델의 Intermediate 분류기에는 Final 분류기의 출력 확률을 목표로 쓰는 KD Loss를 추가합니다.

In [ ]:
# 두 Classifier를 정답 레이블로 학습한다. Self KD 조건에서는 Intermediate에 추가 KD Loss를
# 적용한다.
def update(model, optimizer, images, labels, distill):
    optimizer.zero_grad(set_to_none=True)
    middle, final = model(images)
    ce_middle = F.cross_entropy(middle, labels)
    ce_final = F.cross_entropy(final, labels)
    loss = ce_middle + ce_final
    # Final 출력은 고정된 Soft Target으로 사용한다. 이 항은 Intermediate 출력의 예측 분포를 맞추는 데
    # 사용된다.
    if distill:
        kd = response_kd(middle, final, TEMPERATURE)
        loss = loss + SELF_WEIGHT * kd
    loss.backward()
    optimizer.step()
    return (ce_middle.item(), ce_final.item(), kd.item()) if distill else None

history = []
start = time.perf_counter()
for images, labels in islice(train_loader, TRAIN_STEPS):
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    update(label_only, optim_label, images, labels, distill=False)
    # 같은 배치로 정답만 학습하는 모델과 Self KD 모델을 각각 갱신한다.
    history.append(update(self_kd, optim_self, images, labels, distill=True))

print(f"{len(history)} batches · {time.perf_counter() - start:.1f} s")
ce_middle, ce_final, kd = history[-1]
print(f"마지막 Self KD 배치: Intermediate CE {ce_middle:.3f} | Final CE {ce_final:.3f} | KD {kd:.3f}")

## 5. 두 Classifier의 정확도 비교

출력의 **Intermediate 정확도**와 **Final 정확도**를 각각 읽습니다. Self KD의 직접적인 학습 목표는 Intermediate 출력에 추가되므로, 먼저 **정답만 학습한 모델의 Intermediate**와 **Self KD 모델의 Intermediate**를 비교합니다.

학습을 64배치로 제한한 결과이므로, 정확도 차이와 함께 두 분류기에 어떤 손실을 적용했는지도 확인합니다.

**코드 셀 8 · Intermediate와 Final 정확도 비교**

같은 테스트 이미지 512장에서 정답 학습용 모델과 Self KD 모델의 Intermediate·Final 분류기 정확도를 각각 출력합니다.

In [ ]:
# 같은 테스트 샘플에서 Intermediate와 Final 정확도를 각각 측정한다.
@torch.inference_mode()
def evaluate_classifiers(model, loader):
    model.eval()
    correct_middle, correct_final, count = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        middle, final = model(images)
        correct_middle += (middle.argmax(1) == labels).sum().item()
        correct_final += (final.argmax(1) == labels).sum().item()
        count += labels.numel()
    return 100 * correct_middle / count, 100 * correct_final / count

for name, model in [("정답만 학습", label_only), ("Self KD", self_kd)]:
    middle, final = evaluate_classifiers(model, test_loader)
    print(f"{name}: Intermediate {middle:.2f}% | Final {final:.2f}%")